In [1]:
from matplotlib import pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

import torch
import numpy as np
from copy import deepcopy
from functools import reduce

from torch import nn
import pickle as pkl
from umap import UMAP

from model import DNN
from run_sim import Config, run_sim, create_data, train_model
from utils import cosine_similarity, get_r_2, vector_angle, flatten_list
from tqdm import tqdm
from utils import alignment_score, calc_PR, calc_NC1
import matplotlib as mpl
from functools import reduce



In [2]:
def calc_h_var(G, D, N, L, sig_X):
    return G**(2*L)*(2*D*sig_X)/(D+N)

In [3]:
C = Config()

C.G = 0.8
C.sig_h_2 = 1e-5
# C.gpu_id=1
# C.seed = 1
C.linear_net = True
C.split_actions = True
# C.allow_backwards = True
C.learning_rate = 0.01
C.L=10
C.print_progress = False
C.length_corridors = [20]*1
# C.input_size = 100
C.max_move = 10
C.hidden_size = sum(C.length_corridors)+2*C.max_move+1 + 1
C.num_epochs *= 100
C.algo_name = 'SGD'
C.loss_fn = nn.MSELoss()
C.bias = False
C.egocentric_movement = True
C.lambda_reg = 0
C.B = 1
C.label_noise = 0
# C.whiten_data = True
# C.fixed_output = False
# C.split_actions = False
# C.allow_backwards = True

# C.min_move = 2
# C.one_hot_actions = True
# C.one_hot_inputs = False

In [ ]:
from itertools import product

params_dict = {
    'B': [0.1, 0.5, 1],
    'label_noise': [0, 0.01, 0.1],
    'max_move': [1, C.length_corridors[0]//2]
}
data_dict_l = []
for B, label_noise, max_move in tqdm(product(params_dict['B'], params_dict['label_noise'], params_dict['max_move'])):
    C.B = B
    C.label_noise = label_noise
    C.max_move = max_move
    device = torch.device(f"cuda:{C.gpu_id}" if torch.cuda.is_available() and use_gpu else "cpu")
    torch.manual_seed(C.seed)
    np.random.seed(C.seed)
    X, y, corridor, loc_X, loc_y, action_taken, dim_l, input_size, output_size, n_actions = create_data(C)

    # if i > 0:
    #     cond = abs(action_taken) <= 1
    #     X = X[cond]
    #     y = y[cond]
    #     loc_y = loc_y[cond]
    #     corridor = corridor[cond]
    #     loc_X = loc_X[cond]
    #     action_taken = action_taken[cond]

    X = torch.tensor(X, dtype=torch.float32).to(device)
    y = torch.tensor(y, dtype=torch.float32).to(device)

    if C.sig_h_2 and C.print_progress:
        C.G = (C.sig_h_2*(X.shape[1]+C.hidden_size)/(2*X.shape[1]*X.var()))**(1/(2*C.L))
        print(f'Changed G to {C.G} to get sig_h_2 = {C.sig_h_2}')
    # Create model
    model = DNN(input_size + n_actions, C.hidden_size, output_size, C.L, C.fixed_output, C.linear_net, C.G, C.bias).to(device)
    # if i == 2:
    #     model.load_state_dict(data_dict_l[0]['model_state'])
    #     for param, param_i in zip(model.parameters(), data_dict_l[0]['initial_weights'].values()):
    #         noise = torch.randn_like(param) * (param.std()*0.1)
    #         # param.data.add_(noise)
    #         param.data *= param_i.std()/param.std()
    initial_weights = deepcopy(model.state_dict())

    loss_l, accuracy_l, hidden_l = train_model(C, X, y, model)
    # Testing
    with torch.no_grad():
        outputs, hidden_states = model(X)
    # print(criterion(outputs, y).item()/y_var)
    hidden = hidden_states[-1].detach().cpu().numpy()
    X_dist = torch.cdist(X, X).cpu().numpy()
    y_dist = torch.cdist(y, y).cpu().numpy()
    hidden_dist = torch.cdist(hidden_states[-1].detach(), hidden_states[-1].detach()).cpu().numpy()
    stay_inds = np.where(action_taken == 0)[0]
    loc_y_corridor = loc_y + (corridor * max(loc_y + 1))
    n_corridors = len(C.length_corridors)

    X_np = X.cpu().numpy()  # Convert to numpy array if X is a torch tensor
    y_np = y.cpu().numpy()  # Convert to numpy array if y is a torch tensor
    h_np = hidden  # Convert to numpy array if hidden is a torch tensor

    data_dict = {
        'X': X,
        'y': y,
        'corridor': corridor,
        'loc_X': loc_X.squeeze(),
        'loc_y': loc_y.squeeze(),
        'action_taken': action_taken,
        'hidden_states': hidden_states,
        'loss_l': loss_l,
        'accuracy_l': accuracy_l,
        'outputs': outputs.cpu().numpy(),
        'hidden_l': hidden_l,
        'model_state': deepcopy(model.state_dict()),
        'initial_weights': initial_weights,
        'X_dist': X_dist,
        'y_dist': y_dist,
        'hidden_dist': hidden_dist,
        'stay_inds': stay_inds,
        'loc_y_corridor': loc_y_corridor,
        'n_corridors': n_corridors,
        'X_np': X_np,
        'y_np': y_np,
        'h_np': h_np,
        'C': C
    }

    data_dict_l.append(data_dict)

with open('data_dict_l.pkl', 'wb') as f:
    pkl.dump(data_dict_l, f)

8it [2:04:38, 978.07s/it]

In [6]:
with open('data_dict_l.pkl', 'rb') as f:
    data_dict_l = pkl.load(f)

In [ ]:
def plot_results(data_dict):
    loc_y = data_dict['loc_y']
    corridor = data_dict['corridor']
    action_taken = data_dict['action_taken']
    loss_l = data_dict['loss_l']
    accuracy_l = data_dict['accuracy_l']
    X_dist = data_dict['X_dist']
    y_dist = data_dict['y_dist']
    hidden_dist = data_dict['hidden_dist']
    indices = np.lexsort((loc_y, corridor))
    indices = indices[action_taken[indices]==0]
    fig, axs = plt.subplots(2, 3, figsize=(10, 5))
    axs[0,0].set_axis_off(); axs[0,2].set_axis_off()
    axs[0,1].plot(loss_l)
    axs[0,1].set_yscale('log')
    axs[0,1].twinx().plot(accuracy_l, 'r')
    axs[0,1].set_title("Loss")
    for var, var_name, ax in zip([X_dist, y_dist, hidden_dist], ['X', 'y', 'hidden'], axs[1]):
        ax.imshow(var[indices][:, indices], cmap='viridis')
        ax.set_title(f'{var_name} distance matrix')
        ax.grid(False)
    plt.tight_layout()
    plt.show()

for data_dict in data_dict_l:
    plot_results(data_dict)

In [ ]:
def plot_pca(data_dict):
    loc_y = data_dict['loc_y']; h_np = data_dict['h_np']

    # model.load_state_dict(data_dict['model_state'])
    # _, hidden = model(data_dict_l[0]['X'])
    # h_np = hidden[-1].detach().cpu().numpy()
    # loc_y = data_dict_l[0]['loc_y']
    
    pca = PCA().fit(h_np)
    X_reduced = pca.transform(h_np)
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    # Add cumulative explained variance ratio in the first row
    ax1 = axs[0]
    ax1.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
    ax1.set_xlabel('Number of Components')
    ax1.set_ylabel('Cumulative Explained Variance')
    ax1.set_title(f'Cumulative Explained Variance Ratio')

    ax1 = axs[1]
    s = ax1.scatter(X_reduced[:,0], X_reduced[:,1], c=loc_y, cmap='coolwarm', alpha=0.7)
    ax1.set_xlabel(f'Component 1')
    ax1.set_ylabel(f'Component 2'),
    ax1.axis('equal')

    ax1 = axs[2]
    ax1.scatter(X_reduced[:,0], loc_y, c=loc_y)

    plt.tight_layout()
    plt.show()

for data_dict in data_dict_l:
    plot_pca(data_dict)

In [14]:
import torch
import torch.nn.functional as F

def forward_pass(X, W_l):
    """Forward pass through a sequence of linear layers without non-linearities."""
    h = X
    for W in W_l:
        h = h @ W.T
    return h  # logits

def compute_loss(y_hat, y):
    return F.mse_loss(y_hat, y)

def flatten_params(W_l):
    """Flatten all weight matrices into a single vector."""
    return torch.cat([w.reshape(-1) for w in W_l])

def unflatten_params(flat_params, shapes):
    """Restore list of weight matrices from a flat parameter vector."""
    W_l = []
    idx = 0
    for shape in shapes:
        numel = torch.prod(torch.tensor(shape)).item()
        W_l.append(flat_params[idx:idx+numel].reshape(shape))
        idx += numel
    return W_l

def compute_full_hessian(X, y, W_l):
    """Compute the full Hessian of the loss w.r.t. all weights in W_l."""
    shapes = [w.shape for w in W_l]
    flat_params = flatten_params(W_l).detach().clone().requires_grad_(True)

    # Reconstruct W_l from flat_params for autograd
    def loss_fn(p):
        W_reconstructed = unflatten_params(p, shapes)
        logits = forward_pass(X, W_reconstructed)
        loss = compute_loss(logits, y)
        return loss

    loss = loss_fn(flat_params)
    grad = torch.autograd.grad(loss, flat_params, create_graph=True)[0]

    H_rows = []
    for i in range(grad.numel()):
        grad2 = torch.autograd.grad(grad[i], flat_params, retain_graph=True)[0]
        H_rows.append(grad2)

    H = torch.stack(H_rows, dim=0)  # Shape: (n_params, n_params)
    return H

In [15]:
def factorize_matrix(M, N):
    D1, D2 = M.shape
    # Compute full SVD
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    rank = np.sum(S > 1e-10)  # numerical rank

    # Generate random orthogonal matrix
    Q = np.random.randn(N, N)
    Q, _ = np.linalg.qr(Q)

    S = np.diag(S)
    # Create random factorization that still reconstructs M
    S_pad = np.concatenate([S, np.zeros([D2, N-D2])], 1)
    A = U @ np.sqrt(S_pad) @ Q
    S_pad = np.concatenate([S, np.zeros([N-D2, D2])], 0)
    B = Q.T @ np.sqrt(S_pad) @ Vt
    return A, B

import numpy as np

def get_AB(X, w1, w2, b, n):
    # Step 1: Compute target matrix
    Y = (X @ w1) @ w2 + np.ones((X.shape[0], 1)) @ b  # (m, c)

    # Step 2: Compute effective Z = X^\dagger Y
    X_dagger = np.linalg.pinv(X)                     # (d, m)
    Z = X_dagger @ Y                                 # (d, c)

    # Step 3: Low-rank SVD factorization
    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    n_max = min(n, min(Z.shape))  # Don't take more components than available
    U_n = np.zeros((U.shape[0], n))  # Initialize with zeros
    S_n = np.zeros((n, n))  # Initialize diagonal matrix with zeros
    Vn = np.zeros((n, Vt.shape[1]))  # Initialize with zeros
    
    # Fill available components
    U_n[:, :n_max] = U[:, :n_max]
    S_n[:n_max, :n_max] = np.diag(np.sqrt(S[:n_max]))
    Vn[:n_max, :] = Vt[:n_max, :]

    A = U_n @ S_n                  # (d, n)
    B = S_n @ Vn                   # (n, c)
    return A, B

def find_matrices(W_target, N, L):
    W_matrices_np = []
    input_dim = W_target.shape[0]
    output_dim = W_target.shape[1]
    
    # First matrix: input dimension to N
    W_matrices_np.append(np.random.randn(input_dim, N) * np.sqrt(2.0/(input_dim + N)))
    
    # Middle matrices: N to N
    for _ in range(L-2):
        W_matrices_np.append(np.random.randn(N, N) * np.sqrt(2.0/(2*N)))
        
    # Last matrix: N to output dimension
    W_matrices_np.append(np.random.randn(N, output_dim) * np.sqrt(2.0/(N + output_dim)))
    
    # Convert to torch tensors for training
    W_matrices = [torch.tensor(W, dtype=torch.float32, requires_grad=True) for W in W_matrices_np]
    
    # Training parameters
    optimizer = torch.optim.Adam(W_matrices, lr=0.01)
    n_steps = 100000

    
    # Training loop
    for step in tqdm(range(n_steps)):
        optimizer.zero_grad()
        
        # Compute product of matrices
        W_product = W_matrices[0]
        for W in W_matrices[1:]:
            W_product = W_product @ W
            
        # Compute loss (Frobenius norm of difference)
        loss = torch.norm(W_product - torch.tensor(W_target, dtype=torch.float32))
        
        loss.backward()
        optimizer.step()
        
        # if step % (n_steps//10) == 0:
        #     print(f"Step {step}, Loss: {loss.item():.6f}")
    
    # Convert back to numpy
    W_matrices_np = [W.detach().numpy().T for W in W_matrices]

    # Verify decomposition
    W_product = W_matrices_np[0].T
    for W in W_matrices_np[1:]:
        W_product = W_product @ W.T

    print("\nDecomposition error:", np.linalg.norm(W_product - W_target))

    return W_matrices_np

def find_matrices_orthogonal(W_target, N, L):
    input_dim = W_target.shape[0]
    output_dim = W_target.shape[1]
    
    A, B = factorize_matrix(W_target, N)

    # First matrix: input dimension to N
    Q_l = [np.linalg.qr(np.random.randn(N, N))[0] for _ in range((L-2)//2)]
    Q_T_l = [Q.T for Q in Q_l]
    W_matrices_np = [A] + flatten_list([[Q, Q_T] for Q, Q_T in zip(Q_l, Q_T_l)]) + [B]
        
    W_matrices_np = [W.T for W in W_matrices_np]

    # Verify decomposition
    W_product = W_matrices_np[0].T
    for W in W_matrices_np[1:]:
        W_product = W_product @ W.T

    print("\nDecomposition error:", np.linalg.norm(W_product - W_target))

    return W_matrices_np

def calc_accuracy(X_np, y_np, W_matrices_np):
    W_effective = reduce(np.matmul, [W.T for W in W_matrices_np])
    
    return ((X_np@W_effective).argmax(1)==(y_np.argmax(1))).mean()

def normalize_W_l(W_l, norm=100):
    theta = np.concatenate([W.flatten() for W in W_l])
    factor = np.linalg.norm(theta)/norm
    return [W/factor for W in W_l]


In [ ]:
for data_dict in tqdm(data_dict_l):
    X_np = data_dict['X_np']; y_np = data_dict['y_np']
    X = data_dict['X']; y = data_dict['y']
    final_weights = data_dict['model_state']

    W_l = [W.detach().numpy() for W in data_dict['model_state'].values()]
    H = compute_full_hessian(X, y, list(final_weights.values())).detach().cpu().numpy()
    data_dict['W_l'] = W_l
    data_dict['H'] = H

with open('data_dict_l.pkl', 'wb') as f:
    pkl.dump(data_dict_l, f)

In [ ]:
exp_types = ['Large A', 'Small A', 'Small A + set initial']
for i, trace in enumerate(trace_H_SGD_l):
    print(f'Synthetic W with {exp_types[i]}: {trace}')

for i, trace in enumerate(trace_H_logistic_l):
    print(f'Logistic W with {exp_types[i]}: {trace}')



In [ ]:
for data_dict in data_dict_l:
    H = data_dict['H']
    X_np = data_dict['X_np']
    y_np = data_dict['y_np']
    W_l = data_dict['W_l']
    trace_H = np.trace(H)
    print(trace_H, compute_trace_of_deep_linear_hessian(W_l, X_np.T, y_np.T))

In [ ]:
L = C.length_corridors[0]
L_start = -L/2; L_end = L/2
A = C.max_move
n_model = 1
Win = np.concatenate([np.linspace(L_start,L_end, L), np.linspace(-A,A, 2*A+1)])[:,None]
Wout = 1/n_model*np.linspace(L_start,L_end, L)[None,:]**n_model
b = -1/(n_model+1)*np.linspace(L_start,L_end, L)[None, :]**(n_model+1)

A,B = get_AB(X_np, Win, Wout, b, C.hidden_size)
W_synthetic = (A@B)
print('Accuracy of tailored solution: ', ((X_np@A@B).argmax(1)==(y_np.argmax(1))).mean())

# Decompose W_logistic into L matrices with intermediate dimension N
L = C.L+1  # Number of layers
N = C.hidden_size  # Intermediate dimension

# Initialize matrices with Xavier initialization
W_l = find_matrices_orthogonal(W_synthetic, C.hidden_size, C.L+1)
W_l = data_dict['W_l']
trace_l = [compute_trace_of_deep_linear_hessian(normalize_W_l(W_l, n), X_np.T, y_np.T) for n in np.arange(1, 100)]